# Motion-S — Colab bootstrap

End-to-end setup for the [Motion-S Hierarchical Text-to-Motion Generation for Sign Language](https://www.kaggle.com/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language) competition.

**What this notebook does**
1. Mounts Google Drive (so weights & data persist across Colab sessions).
2. Uploads your `kaggle.json` and configures the Kaggle CLI.
3. Downloads only the small competition files (~few hundred MB) — CSVs, the frozen RVQ-VAE checkpoint, length estimator, baseline notebook, and evaluation script. The 37 GB BVH/NPY raw motion is **not** needed because tokens are already in `train.csv`.
4. Clones the project scaffold and installs deps.
5. Runs the submission validator smoke test, the dataset reconnaissance script, and freezes the 90/10 train/val split.

**Before running**: in your browser, accept the competition rules on the [Rules tab](https://www.kaggle.com/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language/rules) once. Otherwise the API will return 403.

## 1. Mount Drive & upload `kaggle.json`

When the upload widget appears, pick your `kaggle.json` (download it from your Kaggle account settings → API → *Create New Token*).

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

uploaded = files.upload()  # select kaggle.json
assert 'kaggle.json' in uploaded, 'You must upload kaggle.json'

## 2. Configure the Kaggle CLI

In [ ]:
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip -q install --upgrade kaggle
!kaggle --version

## 3. Download competition data

Stored in Drive at `MyDrive/motion-s/data` so it persists. Change `DATA_DIR` if you want a different location.

In [ ]:
import os
COMP = 'motion-s-hierarchical-text-to-motion-generation-for-sign-language'
DATA_DIR = '/content/drive/MyDrive/motion-s/data'
os.environ['COMP'] = COMP
os.environ['DATA_DIR'] = DATA_DIR
!mkdir -p "$DATA_DIR"
!ls -lh "$DATA_DIR"

In [ ]:
# Try per-file download first (cheaper, faster). If any file 403s with
# 'must download all', fall back to the bundle download in the next cell.
FILES = [
    'train.csv',
    'test.csv',
    'sample_submission.csv',
    'rvq_vae_best.pth',
    'length_estimator.pth',
    'evaluation_script.py',
    'baseline_notebook.ipynb',
]
for f in FILES:
    !kaggle competitions download -c "$COMP" -f "{f}" -p "$DATA_DIR"

In [ ]:
# Fallback (uncomment if per-file failed). Downloads the full bundle (~37 GB).
# !kaggle competitions download -c "$COMP" -p "$DATA_DIR"

In [ ]:
# Unzip anything that arrived zipped, then list what we have.
import glob, os, zipfile
for z in glob.glob(f'{DATA_DIR}/*.zip'):
    print('unzipping', z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA_DIR)
    os.remove(z)
!ls -lh "$DATA_DIR"

## 4. Get the project scaffold

Two options — pick **one**:

**A. From GitHub** (preferred). Set `GITHUB_REPO` below to your fork.

In [ ]:
GITHUB_REPO = ''  # e.g. 'https://github.com/<you>/motion-s.git'
PROJECT_DIR = '/content/motion-s'
if GITHUB_REPO:
    !rm -rf "$PROJECT_DIR"
    !git clone "$GITHUB_REPO" "$PROJECT_DIR"
    %cd $PROJECT_DIR
    !rm -rf data && ln -s "$DATA_DIR" data
    !pip -q install -r requirements.txt
else:
    print('GITHUB_REPO is empty — use the upload-zip cell below instead.')

**B. Upload zipped scaffold** (if you don't have it on GitHub yet). On your Windows box:

```powershell
Compress-Archive -Path 'E:\Kaggle\Motion-S Text-to-Sign Motion Generation Signvrse\*' -DestinationPath motion-s.zip
```

Then run the next cell and select `motion-s.zip`.

In [ ]:
from google.colab import files
import zipfile, os, shutil
PROJECT_DIR = '/content/motion-s'
if not os.path.isdir(PROJECT_DIR):
    up = files.upload()  # pick motion-s.zip
    name = next(iter(up))
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    os.makedirs(PROJECT_DIR, exist_ok=True)
    with zipfile.ZipFile(name) as zf:
        zf.extractall(PROJECT_DIR)
    os.remove(name)
    %cd $PROJECT_DIR
    !rm -rf data && ln -s "$DATA_DIR" data
    !pip -q install -r requirements.txt

## 5. Verify everything works

In [ ]:
%cd /content/motion-s
!python -m scripts.test_validator

In [ ]:
# Phase-0 reconnaissance: text/length stats, codebook usage, RVQ-VAE checkpoint shapes.
# Writes data/data_recon.json — paste the rvq_vae_checkpoint section back to me.
!python -m scripts.inspect_data

In [ ]:
# Freeze the stratified 90/10 train/val split.
!python -m scripts.make_split

## 6. GPU check & next steps

In [ ]:
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

**Next**: send back the contents of `data/data_recon.json` (especially the `rvq_vae_checkpoint` block) so we can wire `FrozenRVQVAE.load()` against the real `state_dict` keys, then start Model A (MoMask) and Model B (T2M-GPT).